In [ ]:
import keras
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

In [ ]:
DATA_DIR = "mu3e_trigger_data"
SIGNAL_DATA_FILE = f"{DATA_DIR}/run42_sig_positions.npy"
BACKGROUND_DATA_FILE = f"{DATA_DIR}/run42_bg_positions.npy"

max_barrel_radius = 86.3
max_endcap_distance = 372.6

In [ ]:
signal_data = np.load(SIGNAL_DATA_FILE)
background_data = np.load(BACKGROUND_DATA_FILE)
from src.utils import cartesian_to_cylindrical, normalize_data
normed_background_data = normalize_data(background_data, type="minmax", feature_range=(0, 1), padding_value=-1)
normed_signal_data = normalize_data(signal_data, type="minmax", feature_range=(0, 1), padding_value=-1)
signal_data_cylindrical = cartesian_to_cylindrical(signal_data)
background_data_cylindrical = cartesian_to_cylindrical(background_data)

In [ ]:
bg_mask = background_data[:, :, 0] != -1
signal_mask = signal_data[:, :, 0] != -1
bg_seq_length = bg_mask.sum(axis=1)
signal_seq_length = signal_mask.sum(axis=1)

background_data = background_data[bg_seq_length > 0]
signal_data = signal_data[signal_seq_length > 0]

In [ ]:
sequence_length = signal_data.shape[1]
input_feature_dim = signal_data.shape[2]
feature_dim = 16

In [ ]:
from src.model.components import SelfAttentionStack, MLP, GenerateMask

encoder_input = keras.Input(
    shape=(sequence_length, input_feature_dim), name="encoder_input"
)
mask = GenerateMask(padding_value=-1)(encoder_input)


input_embedding = MLP(
    feature_dim, num_layers=3, hidden_activation="relu", activation="linear", name="input_embedding_mlp"
)(encoder_input)

attention_output = SelfAttentionStack(
    num_heads=4,
    key_dim=feature_dim,
    dropout_rate=0.1,
    stack_size=3,
    name="self_attention_stack",
)(input_embedding, mask)

output_embedding = MLP(
    feature_dim, num_layers=3, hidden_activation="relu", activation="linear", name="output_embedding"
)(attention_output)

encoded_output = keras.layers.GlobalAveragePooling1D(name="global_average_pooling")(
    output_embedding, mask=mask
)

encoder = keras.Model(
    inputs=encoder_input,
    outputs=encoded_output,
    name="encoder",
)

In [ ]:
print(encoder.trainable_variables)
encoder.trainable = False
print(encoder.trainable_variables)

In [ ]:
lstm_encoder_input = keras.Input(
    shape=(sequence_length, feature_dim), name="lstm_encoder_input"
)
lstm_mask = GenerateMask(padding_value=-1)(lstm_encoder_input)
lstm_embedding = MLP(
    feature_dim, num_layers=3, hidden_activation="relu", activation="linear", name="lstm_embedding_mlp"
)(lstm_encoder_input)
lstm_layer_1 = keras.layers.LSTM(
    feature_dim, return_sequences=True, name="lstm_layer_1"
)(lstm_embedding, mask=lstm_mask)
lstm_layer_2 = keras.layers.LSTM(
    feature_dim, return_sequences=True, name="lstm_layer_2"
)(lstm_layer_1, mask=lstm_mask)
lstm_output = keras.layers.Bidirectional(
    keras.layers.LSTM(feature_dim, return_sequences=False, name="lstm_output")
)(lstm_layer_2, mask=lstm_mask)

lstm_encoder = keras.Model(
    inputs=lstm_encoder_input,
    outputs=lstm_output,
    name="lstm_encoder",
)


In [ ]:
print(lstm_encoder.trainable_variables)
lstm_encoder.trainable = False
print(lstm_encoder.trainable_variables)

In [ ]:
from src.model import AutoEncoder
autoencoder = AutoEncoder(
    input_size = feature_dim,
    latent_size =  feature_dim //2,
    num_layers = 3
)(encoder_input)

In [ ]:
print(autoencoder.trainable_variables)
autoencoder.trainable = False
print(autoencoder.trainable_variables)

In [ ]:
from src.training import MultiObjectiveTrainer
from src.training import VarianceCovarianceLoss

trainer = MultiObjectiveTrainer(
    encoder=encoder,
    autoencoder=autoencoder,
    lambda_var=1.0,
    variance_loss = VarianceCovarianceLoss(cov_penalty= 1/25.)
)
encoder_optimizer = keras.optimizers.Adam(learning_rate=0.001)
ae_optimizer = keras.optimizers.Adam(learning_rate=0.01)

In [ ]:
from sklearn.model_selection import train_test_split

bg_train, bg_test = train_test_split(background_data_cylindrical[:10000], test_size=0.2, random_state=42)

bg_train = tf.data.Dataset.from_tensor_slices(bg_train).batch(512, drop_remainder=True)
latent_dataset = bg_train.map(encoder, num_parallel_calls=tf.data.AUTOTUNE)


In [ ]:
for var_epoch in range(10):
    print(f"  Variance Epoch {var_epoch + 1}")
    losses = trainer.train_encoder_variance_step(bg_train, encoder_optimizer)
    print(f"    Losses: {losses}")

for epoch in range(5):
    print(f"Epoch {epoch + 1}")
    print(f"  Autoencoder Epochs")
    losses = trainer.train_autoencoder_step(bg_train, ae_optimizer, num_steps=10)
    for loss in losses:
        print(f"    Loss: {loss}")

    for recon_epoch in range(4):
        print(f"  Reconstruction Epoch {recon_epoch + 1}")
        losses = trainer.train_encoder_step(bg_train, encoder_optimizer)
        print(f"    Losses: {losses}")

In [ ]:
trainer.train_autoencoder_step(bg_train, ae_optimizer, num_steps=5)

In [ ]:
bg_test_embeddings = encoder.predict(bg_test)
signal_embeddings = encoder.predict(signal_data_cylindrical[:2000])
bg_train_embeddings = encoder.predict(bg_train)

In [ ]:
bg_test_diff = bg_test_embeddings - autoencoder.predict(bg_test_embeddings)
signal_diff = signal_embeddings - autoencoder.predict(signal_embeddings)
bg_train_diff = bg_train_embeddings - autoencoder.predict(bg_train_embeddings)

In [ ]:
fig, ax_array = plt.subplots(figsize=(10, 5), nrows=4, ncols=4)
for i in range(16):
    ax = ax_array[i // 4, i % 4]
    bg_ad_score = bg_test_embeddings[:, i]
    signal_ad_score = signal_embeddings[:, i]
    bins = np.linspace(
        min(np.min(bg_ad_score), np.min(signal_ad_score)),
        max(np.max(bg_ad_score), np.max(signal_ad_score)),
        30,
    )
    ax.hist(
        bg_ad_score,
        bins=bins,
        alpha=0.5,
        label="Background Data",
        color="blue",
        density=True,
    )
    ax.hist(
        signal_ad_score,
        bins=bins,
        alpha=0.5,
        label="Signal Data",
        color="red",
        density=True,
    )
    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel(" $\sigma_{"+ f"{i+1}" +"}$")
    ax.set_ylabel("Density")
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2)
fig.savefig("ad_scores_train_test.png", dpi=300, bbox_inches="tight")

In [ ]:
fig, ax_array = plt.subplots(figsize=(10, 5), nrows=4, ncols=4)
for i in range(16):
    ax = ax_array[i // 4, i % 4]
    bg_ad_score = bg_test_diff[:, i]
    signal_ad_score = signal_diff[:, i]
    bins = np.linspace(
        min(np.min(bg_ad_score), np.min(signal_ad_score)),
        max(np.max(bg_ad_score), np.max(signal_ad_score)),
        30,
    )
    ax.hist(
        bg_ad_score,
        bins=bins,
        alpha=0.5,
        label="Background Data",
        color="blue",
        density=True,
    )
    ax.hist(
        signal_ad_score,
        bins=bins,
        alpha=0.5,
        label="Signal Data",
        color="red",
        density=True,
    )
    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel(" $\sigma_{"+ f"{i+1}" +"}$")
    ax.set_ylabel("Density")
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2)
fig.savefig("ad_scores_train_test.png", dpi=300, bbox_inches="tight")

In [ ]:
fig, ax_array = plt.subplots(figsize=(10, 5), nrows=4, ncols=4)
for i in range(16):
    ax = ax_array[i // 4, i % 4]
    bg_ad_score = bg_test_diff[:, i]
    signal_ad_score = bg_train_diff[:, i]
    bins = np.linspace(
        min(np.min(bg_ad_score), np.min(signal_ad_score)),
        max(np.max(bg_ad_score), np.max(signal_ad_score)),
        20,
    )
    ax.hist(
        bg_ad_score,
        bins=bins,
        alpha=0.5,
        label="Test Data",
        color="blue",
        density=True,
    )
    ax.hist(
        signal_ad_score,
        bins=bins,
        alpha=0.5,
        label="Train Data",
        color="red",
        density=True,
    )
    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel(" $\sigma_{"+ f"{i+1}" +"}$")
    ax.set_ylabel("Density")
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2)
fig.savefig("ad_scores_train_test.png", dpi=300, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bg_ad_score = np.linalg.norm(bg_test_diff, axis=1)
signal_ad_score = np.linalg.norm(signal_diff, axis=1)
bins = np.linspace(
    min(np.min(bg_ad_score), np.min(signal_ad_score)),
    max(np.max(bg_ad_score), np.max(signal_ad_score)),
    30,
)
ax.hist(
    bg_ad_score,
    bins=bins,
    alpha=0.5,
    label="Background Data",
    color="blue",
    density=True,
)
ax.hist(
    signal_ad_score,
    bins=bins,
    alpha=0.5,
    label="Signal Data",
    color="red",
    density=True,
)

ax.set_xlabel(f" $\sigma_{i+1}$")
ax.set_ylabel("Density")
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2)
fig.savefig("ad_scores_train_test.png", dpi=300, bbox_inches="tight")

In [ ]:
mu = np.mean(bg_train_diff, axis=0)
sigma = np.std(bg_train_diff, axis=0)

dummy_data = np.random.normal(loc=mu, scale=sigma, size=(1000, feature_dim))
dummy_data_diff = dummy_data - autoencoder.predict(dummy_data)
np.mean(np.linalg.norm(dummy_data_diff, axis=1))
np.mean(np.linalg.norm(bg_test_diff, axis=1))

In [ ]:
fig, ax_array = plt.subplots(figsize=(10, 5), nrows=4, ncols=4)
for i in range(16):
    ax = ax_array[i // 4, i % 4]
    bg_ad_score = bg_test_diff[:, i]
    signal_ad_score = dummy_data_diff[:, i]
    bins = np.linspace(
        min(np.min(bg_ad_score), np.min(signal_ad_score)),
        max(np.max(bg_ad_score), np.max(signal_ad_score)),
        20,
    )
    ax.hist(
        bg_ad_score,
        bins=bins,
        alpha=0.5,
        label="Test Data",
        color="blue",
        density=True,
    )
    ax.hist(
        signal_ad_score,
        bins=bins,
        alpha=0.5,
        label="Dummy Data",
        color="red",
        density=True,
    )
    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel(" $\sigma_{"+ f"{i+1}" +"}$")
    ax.set_ylabel("Density")
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2)
fig.savefig("ad_scores_train_test.png", dpi=300, bbox_inches="tight")